Configurations

In [1]:
import yaml
import os
import datetime as dt
import numpy as np

In [ ]:
folder_path = "data/torchc_amp/"

#- Train configuration -#

# Training configuration dict
train_config = {
    'execution_type': "noiseless", # 'noiseless', 'noisy' or 'real'
    'n_qubits': 4,
    'seed': 0,
    'run_id': 0, # For different circuits or training parameters
    
    # Data
    'reset_config': True, # Reset configuration dicts
    'reset_training_data': True, # Reset training data
    'create_circuits': True, # Create circuits or load from file
    'reset_backend': True, # Get current backend state or load from file

    # Training parameters
    'max_iterations': 1000,
    'gen_iterations': 1,
    'disc_iterations': 3,
    'print_progress_iterations': 1,

    # Compute methods
    'device': "CPU", # 'GPU' or 'CPU'
    'gradient_method': "PSR", # qiskit_algorithms.gradients For now: 'PSR', 'SPSA' and 'REG'

    # Embedding
    'reset_dataset': False, 
    'batch_size': 4, # How many samples' gradients are going to be calculated in a step
    #'real_rate': 0.5, # Rate of training discriminator with real samples instead of generated samples TODO
    'random_input': True, # Add randomness in the input when generating a sample
    'randomness': 0.1 # Multiplier for random values to be less random
}



#- Backend configuration -#

# Get shots from precision
def get_shots(precision):
    if np.isclose(precision, 0, atol=1e-8):
        return None
    else:
        return int(1/(precision**2))


# Backend configuration
backend_config = {
    # Real backend
    'name': "ibm_basquecountry",
    'channel': "ibm_quantum_platform",

    # Noisy backend
    'reset_backend': False, # Get current backend state or load from file
    'timestamp': dt.datetime.now(),

    # Noiseless/Noiseless backend
    'sim_options': {
        'method': 'automatic', # If None, use defalut options (los de abajo)
        'precision': "double", # 'single' or 'double'. Also used to choose torch dtype
        'seed_simulator': train_config['seed']
    },
}

# Backend predefined configuration for GPU cuda execution
if train_config["device"] == "GPU":
    backend_config['sim_options'].update({
        'device': 'GPU', # For nvidia cuda

        'cuStateVec_enable': True,   # NVIDIA library optimization
        'batched_shots_gpu': True,   # Parallelize batch on GPU
        'blocking_enable': False,     # Disable chunking; simulation fits in VRAM 
        #'target_gpus':[0,1], # TODO import torch after? In .py read how many after printing aviables?

        'runtime_parameter_bind_enable': True, # tells Aer to keep the circuit parameterized and bind the numeric values at execution time TODO prueba pa saber las mejores options
    })

# Backend predefined configuration for noise
if train_config["execution_type"] in ["noisy", "real"]:
    backend_config['sim_options']['method'] = 'density_matrix'
    backend_config['train_precision'] = 0.015625
    backend_config['eval_precision'] = 0.015625 # Fully TorchConnector: precision cannot be 0 because of Aer SamplerV2
else:
    backend_config['sim_options']['method'] = 'statevector'
    backend_config['train_precision'] = 0.0
    backend_config['eval_precision'] = 0.0 # Not fully TorchConnector: precision cannot be 0 because of Aer SamplerV2
backend_config['sim_options']['shots'] = get_shots(backend_config["train_precision"])

# Eval backend configuration
backend_config['eval_options'] = backend_config['sim_options'].copy()
if backend_config['eval_precision'] == 0.0:
    backend_config['eval_options']['method'] = 'statevector'
    backend_config['eval_options']['shots'] = get_shots(backend_config["eval_precision"])
'''
method:
    For noiseless execution:
        - statevector
        - matrix_product_state: more qubits, low entanglement
    For noisy execution:
        - density_matrix
        - statevector: uses less memory
    - stabilizer: only for Clifford circuits (only H, CNOT, and S gates)
    - tensor_network: for large circuits (when reaching memory limits, only GPU)
'''

'''
# More options
backend_for_info = AerSimulator()
print("AerSimulator backend configuration options:")
for option in backend_for_info.options:
    print(" -", option)

'''

'\n# More options\nbackend_for_info = AerSimulator()\nprint("AerSimulator backend configuration options:")\nfor option in backend_for_info.options:\n    print(" -", option)\n\n'

In [6]:
#- Create configuration file -#

# Create data_path
train_config["data_path"] = (
    f"{folder_path}"
    f"q{train_config['n_qubits']}_"
    f"{train_config['execution_type']}_"
    f"{train_config['device']}_"
    f"{train_config['gradient_method']}_"
    f"seed{train_config['seed']}_"
    f"id{train_config['run_id']}/"
)

if not os.path.exists(train_config['data_path']):
    os.makedirs(train_config['data_path'])


# Save config file
def create_config_file(filename):
    data = {
            'train_config': train_config,
            'backend_config': backend_config
        }
    
    with open(filename, "w") as file:
        yaml.dump(data, file, default_flow_style=False, sort_keys=False)

    print("Configuration file created. Path: " + filename)


create_config_file(train_config['data_path'] + "config.yaml")

Configuration file created. Path: data/torchc_amp/q4_noiseless_CPU_PSR_seed0_id0/config.yaml


In [7]:
#- Load configuration file -#

# Load config file
def load_config_file(filename):
    if train_config['reset_config'] or not os.path.exists(filename):
        create_config_file(filename)

    with open(filename, "r") as file:
        data = yaml.safe_load(file)

    train_c = data['train_config']
    backend_c = data['backend_config']

    return train_c, backend_c


train_config, backend_config = load_config_file(train_config['data_path'] + "config.yaml")

print(train_config)
print(backend_config)

Configuration file created. Path: data/torchc_amp/q4_noiseless_CPU_PSR_seed0_id0/config.yaml
{'execution_type': 'noiseless', 'n_qubits': 4, 'seed': 0, 'run_id': 0, 'reset_config': True, 'reset_training_data': True, 'create_circuits': True, 'reset_backend': True, 'max_iterations': 1000, 'gen_iterations': 1, 'disc_iterations': 3, 'print_progress_iterations': 1, 'device': 'CPU', 'gradient_method': 'PSR', 'reset_dataset': False, 'batch_size': 4, 'random_input': True, 'data_path': 'data/torchc_amp/q4_noiseless_CPU_PSR_seed0_id0/'}
{'name': 'ibm_basquecountry', 'channel': 'ibm_quantum_platform', 'reset_backend': False, 'timestamp': datetime.datetime(2026, 5, 13, 2, 44, 26, 361472), 'sim_options': {'method': 'statevector', 'precision': 'double', 'seed_simulator': 0, 'shots': None}, 'train_precision': 0.0, 'eval_precision': 0.0, 'eval_options': {'method': 'statevector', 'precision': 'double', 'seed_simulator': 0, 'shots': None}}


In [ ]:
#- Create configuration battery -#

IMPLEMENTATIONS = ("torchc", "torchc_ang", "torchc_amp")
QUBIT_NUMBERS = (4, 8, 16)
EXECUTION_TYPES = ("noiseless", "noisy", "real")
DEVICES = ("CPU", "GPU")
GRADIENT_METHODS = ("PSR", "SPSA", "REG")
RANDOM_INPUTS = {
    "torchc": (None,),
    "torchc_ang": (True, False),
    "torchc_amp": (True, False),
}


def build_train_config(
    implementation,
    n_qubits,
    execution_type,
    device,
    gradient_method,
    random_input=None,
):
    new_train_config = {
        'execution_type': execution_type,
        'n_qubits': n_qubits,
        'seed': 0,
        'run_id': 0,

        # Data
        'reset_config': True,
        'reset_training_data': True,
        'create_circuits': True,
        'reset_backend': True,

        # Training parameters
        'max_iterations': 1000,
        'gen_iterations': 1,
        'disc_iterations': 3,
        'print_progress_iterations': 1,

        # Compute methods
        'device': device,
        'gradient_method': gradient_method,
    }

    if implementation in ("torchc_ang", "torchc_amp"):
        new_train_config.update({
            'reset_dataset': False,
            'batch_size': 4,
            'random_input': random_input,
        })

    data_path_parts = [
        f"data/{implementation}/",
        f"q{new_train_config['n_qubits']}_",
        f"{new_train_config['execution_type']}_",
        f"{new_train_config['device']}_",
        f"{new_train_config['gradient_method']}_",
    ]
    if random_input is not None:
        data_path_parts.append(f"random{random_input}_")
    data_path_parts.extend([
        f"seed{new_train_config['seed']}_",
        f"id{new_train_config['run_id']}/",
    ])
    new_train_config["data_path"] = "".join(data_path_parts)

    return new_train_config


def build_backend_config(new_train_config):
    new_backend_config = {
        # Real backend
        'name': "ibm_basquecountry",
        'channel': "ibm_quantum_platform",

        # Noisy backend
        'reset_backend': False,
        'timestamp': dt.datetime.now(),

        # Noiseless backend
        'sim_options': {
            'method': 'automatic',
            'precision': "double",
            'seed_simulator': new_train_config['seed'],
        },
    }

    if new_train_config["device"] == "GPU":
        new_backend_config['sim_options'].update({
            'device': 'GPU',
            'cuStateVec_enable': True,
            'batched_shots_gpu': True,
            'blocking_enable': False,
            'runtime_parameter_bind_enable': True,
        })

    if new_train_config["execution_type"] in ["noisy", "real"]:
        new_backend_config['sim_options']['method'] = 'density_matrix'
        new_backend_config['train_precision'] = 0.015625
        new_backend_config['eval_precision'] = 0.015625
    else:
        new_backend_config['sim_options']['method'] = 'statevector'
        new_backend_config['train_precision'] = 0.0
        new_backend_config['eval_precision'] = 0.0
    new_backend_config['sim_options']['shots'] = get_shots(new_backend_config["train_precision"])

    new_backend_config['eval_options'] = new_backend_config['sim_options'].copy()
    if new_backend_config['eval_precision'] == 0.0:
        new_backend_config['eval_options']['method'] = 'statevector'
        new_backend_config['eval_options']['shots'] = get_shots(new_backend_config["eval_precision"])

    return new_backend_config


def create_config_file_from_configs(filename, train_c, backend_c):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    data = {
        'train_config': train_c,
        'backend_config': backend_c,
    }

    with open(filename, "w") as file:
        yaml.dump(data, file, default_flow_style=False, sort_keys=False)


created_config_files = []
for implementation in IMPLEMENTATIONS:
    for n_qubits in QUBIT_NUMBERS:
        for execution_type in EXECUTION_TYPES:
            for device in DEVICES:
                for gradient_method in GRADIENT_METHODS:
                    for random_input in RANDOM_INPUTS[implementation]:
                        train_c = build_train_config(
                            implementation=implementation,
                            n_qubits=n_qubits,
                            execution_type=execution_type,
                            device=device,
                            gradient_method=gradient_method,
                            random_input=random_input,
                        )
                        backend_c = build_backend_config(train_c)
                        filename = train_c['data_path'] + "config.yaml"
                        create_config_file_from_configs(filename, train_c, backend_c)
                        created_config_files.append(filename)

print(f"{len(created_config_files)} configuration files created.")